In [9]:
%pip install numpy pandas scipy scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/cc/d5/2b5148f2279196775e1db2aeb85d14b70ac80e7e32b3b28e7ebeafb0901d/scikit_learn-1.9.0-cp312-cp312-macosx_12_0_arm64.whl.metadata
  Obtaining dependency information for joblib>=1.4.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/48/ca/36339329c4604adbcc99c899b7eb1ce1a555c499b6a6860757dc9bfed36d/narwhals-2.22.1-py3-none-any.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.5.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68726a4ae5f2d82ddd9/threadpoolctl-3.6.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 17.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 k

In [8]:
import sys; print(sys.executable)


/usr/local/bin/python3


In [10]:
%matplotlib inline
import numpy as np
import pandas as pd
import pickle
import math
import scipy.signal
from scipy.stats import skew
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter

#plt.rcParams['image.aspect'] = 'auto'
#plt.rcParams['font.size'] = 14

## Cleaning Data

In [14]:
zfile = np.load('../data/processed_data.zip')
data = pd.read_csv('../data/master_processed_no_duplicates_manual_ds.csv')

In [15]:
data.head()

,Difference_R,Difference_L,UIN,Year,Human_Type_R_audiologist,Human_Type_L_audiologist,Human_Type_R_layman,Human_Type_L_layman,Otoscopy - Right ear: (choice=Normal),Otoscopy - Right ear: (choice=Non-occluding cerumen),...,R_layman_manual_read,L_layman_manual_read,TPP_R_audiologist_manual,ECV_R_audiologist_manual,TPP_L_audiologist_manual,ECV_L_audiologist_manual,TPP_R_layman_manual,ECV_R_layman_manual,TPP_L_layman_manual,ECV_L_layman_manual
0,NaN,NaN,1010001,Year 1,A,A,A,A,False,False,...,True,True,-56.0,1.21,-9.0,1.14,-55.0,1.47,6.0,1.42
1,D,D,1010001,Year 2,C (<200 daPa),C (<200 daPa),B,B,False,False,...,True,True,-199.0,1.65,-224.0,1.21,NaN,1.60,-156.0,1.19
2,NaN,NaN,1010002,Year 1,A,A,A,A,False,True,...,True,True,10.0,0.81,1.0,1.09,16.0,1.04,10.0,1.32
3,NaN,NaN,1010002,Year 2,A,A,A,A,False,True,...,True,True,-14.0,1.27,-49.0,1.26,20.0,0.88,33.0,1.34
4,D,NaN,1010003,Year 2,A,A,B,A,False,False,...,True,True,-7.0,0.86,-32.0,0.98,NaN,0.05,-8.0,0.59


In [16]:
print(f'{len(np.unique(data["UIN"]))} unique UIN')

classes = ['A', 'B', 'C']
features = ['TPP', 'ECV', 'Type']

frames = []
for ear in ['R', 'L']:
    names = ['UIN', f'Difference_{ear}']
    renames = ['UIN', 'Difference']

    for reader in ['audiologist', 'layman']:
        names.append(f'Human_Type_{ear}_{reader}')
        renames.append(f'Human_Type_{reader}')

        names.append(f'Date_{reader}')
        renames.append(f'Date_{reader}')

        names.append(f'{ear}_test_idx_{reader}')
        renames.append(f'test_idx_{reader}')

        names.extend([f'{feat}_{ear}_{reader}' for feat in features])
        renames.extend([f'{feat}_{reader}' for feat in features])

        names.extend([f'{feat}_{ear}_{reader}_manual' for feat in features[:2]])
        renames.extend([f'{feat}_{reader}_manual' for feat in features[:2]])

    df = data[names]
    df = df.rename(columns=dict(zip(names, renames)))

    for reader in ['audiologist', 'layman']:
        def get_data_name(x):
            name = f'{x["UIN"]}_'+x[f"Date_{reader}"][:10]+f'_{reader}_{ear}'
            return name

        df[f'data_name_{reader}'] = df.apply(get_data_name, axis=1)
        df = df.drop(columns=f'Date_{reader}')

    frames.append(df)

frames = pd.concat(frames, ignore_index=True)
frames = frames.replace({
    'C  (<200 daPa)': 'C',
    'TYPE_A': 'A',
    'TYPE_AS': 'A',
    'TYPE_AD': 'A',
    'TYPE_B': 'B',
    'TYPE_C': 'C',
})

print(f'{2*len(frames)} intial tracings')

# Remove no test_idx
has_both_test_idx = (frames['test_idx_audiologist'].notna() & frames['test_idx_layman'].notna())
print(f'{(~has_both_test_idx).sum()} ears without both test index')
frames = frames[has_both_test_idx]
frames.reset_index(drop=True, inplace=True)

# Remove no machine type
has_both_machine_type = (frames['Type_audiologist'].isin(classes) & frames['Type_layman'].isin(classes))
print(f'{(~has_both_machine_type).sum()} ears without both machine type')
frames = frames[has_both_machine_type]
frames.reset_index(drop=True, inplace=True)

# Remove CNE evals
# inital team only removed audiologist CNEs
has_both_human_type = (frames['Human_Type_audiologist'].isin(classes) & frames['Human_Type_layman'].isin(classes))
print(f'{(~has_both_human_type).sum()} ears with non-CNE class')
frames = frames[has_both_human_type]
frames.reset_index(drop=True, inplace=True)

print(f'{len(np.unique(frames["UIN"]))} Unique UIN after removing bad entries')
print(f'{len(frames)} total ears')
print(f'{2*len(frames)} usable tracings')

# Check tracings are in zip file
datafile_names = [f[15:] for f in zfile.files[1:]]
assert frames['data_name_audiologist'].isin(datafile_names).all()
assert frames['data_name_layman'].isin(datafile_names).all()

1583 unique UIN
9968 intial tracings
156 ears without both test index
7 ears without both machine type
43 ears with non-CNE class
1574 Unique UIN after removing bad entries
4778 total ears
9556 usable tracings


In [17]:
frames.head()

,UIN,Difference,Human_Type_audiologist,test_idx_audiologist,TPP_audiologist,ECV_audiologist,Type_audiologist,TPP_audiologist_manual,ECV_audiologist_manual,Human_Type_layman,test_idx_layman,TPP_layman,ECV_layman,Type_layman,TPP_layman_manual,ECV_layman_manual,data_name_audiologist,data_name_layman
0,1010001,NaN,A,2.0,-56.0,1.21,A,-56.0,1.21,A,1.0,-55.0,1.47,A,-55.0,1.47,1010001_2018-04-02_audiologist_R,1010001_2018-04-02_layman_R
1,1010001,D,C,0.0,-199.0,1.65,A,-199.0,1.65,B,0.0,NaN,1.60,B,NaN,1.60,1010001_2019-03-19_audiologist_R,1010001_2019-03-19_layman_R
2,1010002,NaN,A,0.0,5.0,0.79,A,10.0,0.81,A,1.0,16.0,1.02,A,16.0,1.04,1010002_2018-04-02_audiologist_R,1010002_2018-04-02_layman_R
3,1010002,NaN,A,0.0,-22.0,1.24,A,-14.0,1.27,A,0.0,15.0,0.88,A,20.0,0.88,1010002_2019-03-19_audiologist_R,1010002_2019-03-19_layman_R
4,1010003,D,A,0.0,-7.0,0.86,A,-7.0,0.86,B,0.0,NaN,0.01,B,NaN,0.05,1010003_2019-03-19_audiologist_R,1010003_2019-03-19_layman_R


# Split Dataset

In [22]:
def count_labels(index_set, name, global_total=None):
    counts = Counter()
    for idx in index_set:
        row = frames.loc[idx]
        for reader in ['audiologist', 'layman']:
            label = row.get(f'Human_Type_{reader}')
            if label in ['A', 'B', 'C']:
                counts[label] += 1

    ordered = {k: counts.get(k, 0) for k in ['A', 'B', 'C']}
    subtotal = sum(ordered.values())

    print(f"\n{name} Set Label Counts:")
    for k in ['A', 'B', 'C']:
        count = ordered[k]
        pct = f"{(100 * count / global_total):.2f}%" if global_total else ""
        print(f"  {k}: {count:,} ({pct})")
    print(f"  Subtotal: {subtotal:,} ({(100 * subtotal / global_total):.2f}%)")

In [20]:
# Seperate totoal dataset into a training set for neuton, a validation set for neuton, and a set to analyse the accuracy of the neuton model
splitter = GroupShuffleSplit(n_splits=1, train_size=0.9, test_size=0.1, random_state=42) # Use GroupShuffleSplit to ensure reproducibility
train_val_idx, analysis_idx = next(splitter.split(frames, groups=frames['UIN']))

frames_trainval = frames.iloc[train_val_idx].copy()
gss_inner = GroupShuffleSplit(n_splits=1, train_size=0.8, test_size=0.2, random_state=99)
train_idx_rel, val_idx_rel = next(gss_inner.split(frames_trainval, groups=frames_trainval['UIN']))

# Absolute indices
analysis_idx = frames.index[analysis_idx]
train_idx = frames_trainval.index[train_idx_rel]
val_idx = frames_trainval.index[val_idx_rel]

In [ ]:
total_labeled = 0
for idx in frames.index:
    for reader in ['audiologist', 'layman']:
        if frames.loc[idx, f'Human_Type_{reader}'] in ['A', 'B', 'C']:
                                     total_labeled += 1

count_labels(train_idx, "Training", global_total=total_labeled)
count_labels(val_idx, "Validation", global_total=total_labeled)
count_labels(analysis_idx, "Analysis", global_total=total_labeled)

NameError: name 'count_labels' is not defined

Formal Dataset for ML Training

In [ ]:
def get_interp(trace):
    if trace[0][0] > trace[0][1]:
        y = np.interp(standard_x, trace[0][::-1], trace[1][::-1])
    elif trace[0][0] < trace[0][1]:
        raise RuntimeError

    y /= 100
    z = sosfiltfilt(lpf, y)
    return np.stack([y, z])

In [ ]:
def extract_trace_features(trace):
    pressure = trace[0]
    compliance = trace[1]

    sa = float(np.max(compliance))
    sa_idx = int(np.argmax(compliance))
    est_tpp = float(pressure[sa_idx])
    est_ecv = compliance[-1]/100

    half_max = sa / 2
    over_half = np.where(compliance >= half_max)[0]
    tw = float(pressure[over_half[-1]] - pressure[over_half[0]] if len(over_half) else 0)

    return {
        "sa": sa,
        "est_tpp": est_tpp,
        "est_ecv": est_ecv,
        "tw": tw
    }

In [ ]:
standard_x = np.arange(-399, 201, step=10, dtype=float)
lpf = butter(4, 0.04, 'lowpass', output='sos')

In [ ]:
rows_train, rows_val, rows_analysis = [], [], []

train_set = set(train_idx)
val_set = set(val_idx)
analysis_set = set(analysis_idx)

for idx, row in frames.iterrows():
    for reader in ['audiologist', 'layman']:
        label = row.get(f'Human_Type_{reader}')
        if label not in label_map:
            continue

        try:
            uin = row['UIN']

            tpp = row.iloc[7 if reader == 'audiologist' else 14] ### TECHDEBT: check difference between TPP_{reader} and TPP_{reader}_manual. Currently using manual
            ecv = row.iloc[8 if reader == 'audiologist' else 15] ### TECHDEBT: check difference between ECV_{reader} and TPP_{reader}_manual. Currently using manual

            features = {
                "uin": uin,
                "idx": idx,
                "tpp": tpp,
                "ecv": ecv
            }

            raw_data = zfile['processed_data/' + row[f'data_name_{reader}'] + '.npy']
            raw_trace = np.stack([raw_data[0], raw_data[1]])
            trace = get_interp(raw_trace)
            features.update(extract_trace_features(raw_trace))

            features.update({f"sample_{i+1}": val for i, val in enumerate(trace[1])})
        except KeyError:
            continue

#pd.DataFrame(rows_train).to_csv("../data/training_data/train_neuton.csv", index=False)
#pd.DataFrame(rows_val).to_csv("../data/training_data/val_neuton.csv", index=False)
#pd.DataFrame(rows_analysis).to_csv("../data/training_data/analysis_neuton.csv", index=False)

#print("Saved train_neuton.csv, val_neuton.csv, and analysis_neuton.csv")